## Subtask 2a: John

### LaBSE embeddings with cosine similarity and a tuned threshold

In [2]:
%pip install -q sentence-transformers scikit-learn

#### Imports

In [3]:
import pandas as pd
import numpy as np
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics import roc_auc_score


#### Load the usages, definitions, and development labels

In [4]:
folder = Path("../data/raw/development-swedish")

usages = pd.read_json(folder / "usages.jsonl", lines=True)
gold = pd.read_json(folder / "subtask2.jsonl", lines=True)

definitions_path = folder / "definitions.jsonl"

if definitions_path.suffix == ".jsonl":
    definitions = pd.read_json(definitions_path, lines=True)
else:
    definitions = pd.read_json(definitions_path)

definitions.head()

,word,definition
0,kvinna,Generell beteckning för kvinnokönet eller samt...
1,skär,"Klar, blank, skir; ren, ljus och vacker; skina..."
2,fröken,"Gift eller ogift yrkesutövande kvinna, företrä..."
3,dumpa,Låta något falla; lägga eller sänka ner
4,exportera,Föra ut en produkt ur ett land (till ett annat...


#### Match each usage with its target definition and label

In [5]:
target_definitions = definitions[["word", "definition"]].copy()

data = gold[["word", "sentence_id", "label"]].merge(
    usages[["word", "sentence_id", "text", "start", "end"]],
    on=["word", "sentence_id"],
    how="left",
    validate="one_to_one"
)

data = data.merge(
    target_definitions,
    on="word",
    how="left",
    validate="many_to_one"
)

if data[["text", "definition"]].isna().any().any():
    raise ValueError("Some usages or target definitions are missing.")

data["label"] = data["label"].astype(int)

if not data["label"].isin([0, 1]).all():
    raise ValueError("Expected binary labels: 0 or 1.")

data = data.reset_index(drop=True)

data[["word", "text", "definition", "label"]].head()

,word,text,definition,label
0,kvinna,nesliga lif medelst sjelfmord ; och ett stort ...,Generell beteckning för kvinnokönet eller samt...,0
1,kvinna,"— Jag sov ; jag väcktes av en kylande känsla ,...",Generell beteckning för kvinnokönet eller samt...,0
2,kvinna,"orden , så mycken fasthet öfver den gamla kvin...",Generell beteckning för kvinnokönet eller samt...,0
3,kvinna,Dessa klubbar ha anordnat sällskapliga samkväm...,Generell beteckning för kvinnokönet eller samt...,0
4,kvinna,Den avvisningshotade eritranska kvinnan Abiba ...,Generell beteckning för kvinnokönet eller samt...,0


#### Create short contexts around the target word

In [6]:
def get_context(row):
    text = row["text"]
    start = int(row["start"])
    end = int(row["end"])

    left = max(0, start - 120)
    right = min(len(text), end + 120)

    return text[left:right].strip()


data["context"] = data.apply(get_context, axis=1)

display(data[["word", "context", "definition"]].head())

,word,context,definition
0,kvinna,nesliga lif medelst sjelfmord ; och ett stort ...,Generell beteckning för kvinnokönet eller samt...
1,kvinna,"— Jag sov ; jag väcktes av en kylande känsla ,...",Generell beteckning för kvinnokönet eller samt...
2,kvinna,"orden , så mycken fasthet öfver den gamla kvin...",Generell beteckning för kvinnokönet eller samt...
3,kvinna,Dessa klubbar ha anordnat sällskapliga samkväm...,Generell beteckning för kvinnokönet eller samt...
4,kvinna,Den avvisningshotade eritranska kvinnan Abiba ...,Generell beteckning för kvinnokönet eller samt...


#### Generate LaBSE embeddings

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

labse = SentenceTransformer(
    "sentence-transformers/LaBSE",
    device=device
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
all_texts = (
    data["context"].tolist()
    + data["definition"].drop_duplicates().tolist()
)

lengths = [
    len(labse.tokenizer.encode(text))
    for text in all_texts
]

In [9]:
context_vectors = labse.encode(
    data["context"].tolist(),
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

definition_texts = data["definition"].drop_duplicates().tolist()

definition_vectors = labse.encode(
    definition_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
)

definition_lookup = dict(
    zip(definition_texts, definition_vectors)
)

paired_definition_vectors = np.stack([
    definition_lookup[text]
    for text in data["definition"]
])

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

#### Calculate similarity scores

In [10]:
data["similarity"] = np.sum(
    context_vectors * paired_definition_vectors,
    axis=1
)

data[["word", "context", "definition", "similarity", "label"]].head()

,word,context,definition,similarity,label
0,kvinna,nesliga lif medelst sjelfmord ; och ett stort ...,Generell beteckning för kvinnokönet eller samt...,0.323493,0
1,kvinna,"— Jag sov ; jag väcktes av en kylande känsla ,...",Generell beteckning för kvinnokönet eller samt...,0.390163,0
2,kvinna,"orden , så mycken fasthet öfver den gamla kvin...",Generell beteckning för kvinnokönet eller samt...,0.415619,0
3,kvinna,Dessa klubbar ha anordnat sällskapliga samkväm...,Generell beteckning för kvinnokönet eller samt...,0.457884,0
4,kvinna,Den avvisningshotade eritranska kvinnan Abiba ...,Generell beteckning för kvinnokönet eller samt...,0.411359,0


#### Separate threshold tuning from evaluation

In [11]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

tune_indices, test_indices = next(
    splitter.split(data, groups=data["word"])
)

tune_data = data.iloc[tune_indices].copy()
test_data = data.iloc[test_indices].copy()

print("Tuning words:", tune_data["word"].unique())
print("Held-out words:", test_data["word"].unique())

Tuning words: ['kvinna' 'fröken' 'dumpa' 'fru' 'förort' 'suga' 'motiv']
Held-out words: ['skär' 'exportera' 'herre']


In [12]:
def average_word_f1(frame, predictions):
    results = frame[["word", "label"]].copy()
    results["prediction"] = np.asarray(predictions)

    scores = []

    for word, group in results.groupby("word"):
        score = f1_score(
            group["label"],
            group["prediction"],
            zero_division=0,
            average="macro"
        )
        scores.append(score)

    return np.mean(scores)

In [13]:
threshold_results = []

for threshold in np.linspace(-1, 1, 201):
    predictions = (
        tune_data["similarity"] >= threshold
    ).astype(int)

    score = average_word_f1(tune_data, predictions)

    threshold_results.append({
        "threshold": threshold,
        "average_f1": score
    })

threshold_results = pd.DataFrame(threshold_results)

best_result = threshold_results.loc[
    threshold_results["average_f1"].idxmax()
]

best_threshold = float(best_result["threshold"])

print("Selected threshold:", best_threshold)
print("Tuning Average F1:", best_result["average_f1"])

Selected threshold: 0.42999999999999994
Tuning Average F1: 0.43074682098836864


#### Evaluate against baseline (all 1s)

In [14]:
test_predictions = (
    test_data["similarity"] >= best_threshold
).astype(int).to_numpy()

baseline_predictions = np.ones(len(test_data), dtype=int)

comparison = []

for name, predictions in [
    ("All-ones baseline", baseline_predictions),
    ("LaBSE + threshold", test_predictions)
]:
    comparison.append({
        "method": name,
        "average_word_f1": average_word_f1(
            test_data, predictions
        ),
        "precision_pooled": precision_score(
            test_data["label"], predictions, zero_division=0
        ),
        "recall_pooled": recall_score(
            test_data["label"], predictions, zero_division=0
        )
    })

pd.DataFrame(comparison)

,method,average_word_f1,precision_pooled,recall_pooled
0,All-ones baseline,0.256071,0.354717,1.000000
1,LaBSE + threshold,0.540761,0.750000,0.702128


## Subtask 2: similarity analysis (ty)

##### loading definitions / right answers

In [15]:
s2 = pd.read_json("../data/raw/development-swedish/subtask2.jsonl", lines=True)
defs = pd.read_json("../data/raw/development-swedish/definitions.jsonl", lines=True)
defs

,word,definition
0,kvinna,Generell beteckning för kvinnokönet eller samt...
1,skär,"Klar, blank, skir; ren, ljus och vacker; skina..."
2,fröken,"Gift eller ogift yrkesutövande kvinna, företrä..."
3,dumpa,Låta något falla; lägga eller sänka ner
4,exportera,Föra ut en produkt ur ett land (till ett annat...
5,fru,"Maka, hustru (oavsett klass)"
6,herre,Manlig överklassperson med betoning på privile...
7,förort,En invandrartät stadsdel som ligger utanför in...
8,suga,Dra in vätska eller gas i munnen genom åstadko...
9,motiv,Ämne för konstnärlig fram­ställning in­om bild...


##### % of "yes" answers per word (6% to 95%)

In [16]:
yes_share = s2.groupby("word")["label"].mean().round(2)
yes_share.sort_values()

,label
word,
fröken,0.06
herre,0.07
motiv,0.18
suga,0.19
kvinna,0.23
skär,0.28
dumpa,0.31
förort,0.49
fru,0.89


##### saving john's labse sim scores so we don't have to rerun the model

In [17]:
!mkdir -p ../outputs/subtask2
scores = data[["word", "sentence_id", "similarity", "label"]]
scores.to_csv("../outputs/subtask2/labse_scores.csv", index=False)
scores.head()

,word,sentence_id,similarity,label
0,kvinna,137d9286_62_69,0.323493,0
1,kvinna,b382a5e9_92_98,0.390163,0
2,kvinna,28ffffe7_42_50,0.415619,0
3,kvinna,3754909a_117_124,0.457884,0
4,kvinna,59ebb00_32_39,0.411359,0


##### do yes sentences score higher than no sentences? (avg similarity per word)

In [18]:
avg = scores.pivot_table(index="word", columns="label", values="similarity").round(3)
avg.columns = ["avg_no", "avg_yes"]
avg["difference"] = avg["avg_yes"] - avg["avg_no"]
avg.sort_values("difference")

,avg_no,avg_yes,difference
word,,,
exportera,0.491,0.478,-0.013
dumpa,0.416,0.408,-0.008
herre,0.358,0.360,0.002
suga,0.399,0.412,0.013
förort,0.311,0.325,0.014
fröken,0.355,0.385,0.030
skär,0.356,0.389,0.033
kvinna,0.409,0.444,0.035
motiv,0.234,0.276,0.042


the gap between yes and no is pretty small, so it's easy for us to conclude that labse similarity barely tells them apart

##### auc per word... chance a yes setnece scores higher than a no  --- 1 would be perfect

In [19]:
rows = []
for word, group in scores.groupby("word"):
    rows.append({"word": word, "labse_auc": roc_auc_score(group["label"], group["similarity"])})

auc_table = pd.DataFrame(rows).round(2)
auc_table.sort_values("labse_auc")

,word,labse_auc
1,exportera,0.43
0,dumpa,0.45
5,herre,0.51
4,förort,0.55
9,suga,0.57
8,skär,0.60
3,fröken,0.65
6,kvinna,0.67
7,motiv,0.69
2,fru,0.71


##### trying diff model: multilingual-E5

In [20]:
e5 = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

#### e5 similarity between each sentence and defiintion

In [21]:
e5_sentences = e5.encode(("passage: " + data["context"]).tolist(), normalize_embeddings=True)
e5_definitions = e5.encode(("query: " + data["definition"]).tolist(), normalize_embeddings=True)

data["e5_similarity"] = (e5_sentences * e5_definitions).sum(axis=1)

##### comparing labse vs e5 (higher auc = better at telling yes from no)

In [22]:
e5_aucs = []
for word, group in data.groupby("word"):
    e5_aucs.append(roc_auc_score(group["label"], group["e5_similarity"]))

auc_table["e5_auc"] = e5_aucs
print("average labse auc:", auc_table["labse_auc"].mean().round(2))
print("average e5 auc:", round(sum(e5_aucs) / len(e5_aucs), 2))
auc_table.round(2)

average labse auc: 0.58
average e5 auc: 0.63


,word,labse_auc,e5_auc
0,dumpa,0.45,0.40
1,exportera,0.43,0.50
2,fru,0.71,0.37
3,fröken,0.65,0.63
4,förort,0.55,0.75
5,herre,0.51,0.74
6,kvinna,0.67,0.74
7,motiv,0.69,0.90
8,skär,0.60,0.62
9,suga,0.57,0.62


##### E5 tells yes and no apart better than LaBSE on average (0.63 vs 0.58), but not for all of them... It helped a lot on motiv, but got much worse on fru

##### relative e5 score... bascially how much higher or lower than word's average

In [23]:
data["e5_relative"] = data["e5_similarity"] - data.groupby("word")["e5_similarity"].transform("mean")

##### gonna do same split as john -- tune on 7 words, test on 3

In [24]:
test_words = list(test_data["word"].unique())
e5_tune = data[~data["word"].isin(test_words)]
e5_test = data[data["word"].isin(test_words)]
print("test words:", test_words)

test words: ['skär', 'exportera', 'herre']


##### trying cutoffs on the 7 tuning words

In [25]:
cutoff_results = []
for cutoff in np.linspace(-0.1, 0.1, 201):
    predictions = (e5_tune["e5_relative"] >= cutoff).astype(int)
    cutoff_results.append({"cutoff": cutoff, "f1": average_word_f1(e5_tune, predictions)})

cutoff_results = pd.DataFrame(cutoff_results)
best_cutoff = cutoff_results.loc[cutoff_results["f1"].idxmax(), "cutoff"]
print("best cutoff:", round(best_cutoff, 3))

best cutoff: 0.01


##### testing on the 3 held out words versuus the every single one is classified as yes baseline

In [26]:
e5_test_predictions = (e5_test["e5_relative"] >= best_cutoff).astype(int)
print("all-yes baseline:", round(average_word_f1(e5_test, np.ones(len(e5_test))), 3))
print("E5:", round(average_word_f1(e5_test, e5_test_predictions), 3))

all-yes baseline: 0.256
E5: 0.475


e5 gets 0.475 vs 0.256 for the all yeses, so it (now) comfortably beats the baseline following the macro tweak in the eval! better auc didn't help since the cutoff from the tuning words doesn't fit the test words (esp exportera, which is 95% yes)

##### comparison w/ right info: e5 with the same  cutoff method as john's labse

In [27]:
raw_results = []
for cutoff in np.linspace(0, 1, 201):
    predictions = (e5_tune["e5_similarity"] >= cutoff).astype(int)
    raw_results.append({"cutoff": cutoff, "f1": average_word_f1(e5_tune, predictions)})

raw_results = pd.DataFrame(raw_results)
best_raw = raw_results.loc[raw_results["f1"].idxmax(), "cutoff"]

e5_raw_predictions = (e5_test["e5_similarity"] >= best_raw).astype(int)
print("best raw cutoff:", round(best_raw, 3))
print("E5 (raw cutoff):", round(average_word_f1(e5_test, e5_raw_predictions), 3))

best raw cutoff: 0.785
E5 (raw cutoff): 0.411


#### Findings: with the new official scoring, both of our models beat the baselines (which are now 0.256 and 0.318). labse works the best (0.541), while e5 outputs 0.475 when we compared each sentence to its word's average, and 0.411 with one cutoff for all words. Although e5 ranks yes/no better, it scores are all bunched together, so one cutoff doesn't work well on new words. we only tested on 3 held out words though

##### making labse predictions for all of the 10 words (ussing the submission format from the comps website)

In [28]:
labse_predictions = data[["word", "sentence_id"]].copy()
labse_predictions["label"] = (data["similarity"] >= best_threshold).astype(int)

!mkdir -p ../outputs/predictions/SV
labse_predictions.to_json("../outputs/predictions/SV/SV_subtask2_labse.jsonl", orient="records", lines=True)
labse_predictions.head()

,word,sentence_id,label
0,kvinna,137d9286_62_69,0
1,kvinna,b382a5e9_92_98,0
2,kvinna,28ffffe7_42_50,0
3,kvinna,3754909a_117_124,1
4,kvinna,59ebb00_32_39,0


##### scoring w/ comp script

In [29]:
import sys
sys.path.insert(0, "evaluation/python-bcubed")
sys.path.insert(0, "evaluation/semeval2027-task3")
from evaluation import evaluate_subtask2

official_score = evaluate_subtask2(
    Path("../data/raw/development-swedish/subtask2.jsonl"),
    Path("../outputs/predictions/SV/SV_subtask2_labse.jsonl")
)
print("official F1 w/ all 10 words:", round(official_score, 3))

official F1 w/ all 10 words: 0.464


##### ran the official scorer on all 10 words: labse gets 0.464, which beats all yes (0.236) and all no (0.361). should be ready to submit to codabench

##### seeing what it looks like per word: labse vs the baselines (macro f1 eval used)

In [30]:
rows = []
for word, group in data.groupby("word"):
    labse_pred = (group["similarity"] >= best_threshold).astype(int)
    rows.append({
        "word": word,
        "held_out": word in test_words,
        "yes_share": group["label"].mean(),
        "all_yes": f1_score(group["label"], [1] * len(group), average="macro", zero_division=0),
        "all_no": f1_score(group["label"], [0] * len(group), average="macro", zero_division=0),
        "labse": f1_score(group["label"], labse_pred, average="macro", zero_division=0),
    })

breakdown = pd.DataFrame(rows).round(2)
breakdown.sort_values("labse")

,word,held_out,yes_share,all_yes,all_no,labse
2,fru,False,0.89,0.47,0.10,0.11
4,förort,False,0.49,0.33,0.34,0.42
7,motiv,False,0.18,0.15,0.45,0.45
0,dumpa,False,0.31,0.24,0.41,0.45
1,exportera,True,0.95,0.49,0.05,0.47
3,fröken,False,0.06,0.05,0.49,0.49
9,suga,False,0.19,0.16,0.45,0.52
5,herre,True,0.07,0.07,0.48,0.54
6,kvinna,False,0.23,0.19,0.44,0.58
8,skär,True,0.28,0.22,0.42,0.61


##### labse beats both baselines on 6 of the 10 words. it does the poorest on fru with just 0.11.. most fru sentences are "yes" but get low scores. proves that one numerical cut off doesnt work for everything

##### alternating test... doing each word as the test word (should be more consistent than just the 3 words)

In [31]:
rows = []

for hidden_word in data["word"].unique():

    other_words = data[data["word"] != hidden_word]
    hidden = data[data["word"] == hidden_word]

    top_cutoff = 0
    top_score = -1
    for cutoff in np.linspace(-1, 1, 201):
        guesses = (other_words["similarity"] >= cutoff).astype(int)
        score = average_word_f1(other_words, guesses)
        if score > top_score:
            top_cutoff = cutoff
            top_score = score

    hidden_guesses = (hidden["similarity"] >= top_cutoff).astype(int)

    rows.append({
        "hidden_word": hidden_word,
        "cutoff": top_cutoff,
        "labse": average_word_f1(hidden, hidden_guesses),
        "all_yes": average_word_f1(hidden, np.ones(len(hidden))),
        "all_no": average_word_f1(hidden, np.zeros(len(hidden))),
    })

rotation = pd.DataFrame(rows)
rotation.round(2)

,hidden_word,cutoff,labse,all_yes,all_no
0,kvinna,0.42,0.53,0.19,0.44
1,skär,0.43,0.61,0.22,0.42
2,fröken,0.45,0.47,0.05,0.49
3,dumpa,0.43,0.45,0.24,0.41
4,exportera,0.45,0.45,0.49,0.05
5,fru,0.43,0.11,0.47,0.10
6,herre,0.43,0.54,0.07,0.48
7,förort,0.45,0.39,0.33,0.34
8,suga,0.45,0.49,0.16,0.45
9,motiv,0.43,0.45,0.15,0.45


In [32]:
rotation[["labse", "all_yes", "all_no"]].mean().round(3)

,0
labse,0.449
all_yes,0.236
all_no,0.361


##### alternating so each word acts a test once... labse averages 0.449 vs 0.236 (all yes) and vs 0.361 (all no)